In [ ]:
!pip install -q openeo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 9.5 MB/s eta 0:00:00


In [ ]:
!pip install -q openeo

In [ ]:
import openeo

connection = openeo.connect(
    "https://openeo.dataspace.copernicus.eu"
)

connection.authenticate_oidc()

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=HGKE-MADU 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [ ]:
lat = 9.966636
lon = 77.429217

print("Latitude:", lat)
print("Longitude:", lon)

Latitude: 9.966636
Longitude: 77.429217


In [ ]:
cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": lon - 0.01,
        "south": lat - 0.01,
        "east": lon + 0.01,
        "north": lat + 0.01
    },
    temporal_extent=["2025-01-01", "2025-12-31"],
    bands=["B03", "B04", "B08"]
)

print(cube)

DataCube(<PGNode 'load_collection' at 0x78c62c6223f0>)


In [ ]:
# NDVI = (B08 - B04) / (B08 + B04)
ndvi = (cube.band("B08") - cube.band("B04")) / (
    cube.band("B08") + cube.band("B04")
)

# NDWI = (B03 - B08) / (B03 + B08)
ndwi = (cube.band("B03") - cube.band("B08")) / (
    cube.band("B03") + cube.band("B08")
)

print("NDVI and NDWI calculations created successfully!")

NDVI and NDWI calculations created successfully!


In [ ]:
# Calculate mean NDVI and NDWI for the selected area
ndvi_mean = ndvi.reduce_dimension(
    dimension="t",
    reducer="mean"
)

ndwi_mean = ndwi.reduce_dimension(
    dimension="t",
    reducer="mean"
)

print("Ready to download NDVI and NDWI results.")

Ready to download NDVI and NDWI results.


In [ ]:
ndvi_mean.download(
    "veerapandi_ndvi_2025.tif",
    format="GTiff"
)

print("NDVI GeoTIFF downloaded successfully!")

OpenEoApiError: [500] Internal: Unexpected error during 'reduce_dimension' (node id 'reducedimension2'): Exception during Spark execution: org.apache.spark.SparkException: Job 554 cancelled because SparkContext was shut down. The process had these arguments: {'data': GeopysparkDataCube(metadata=GeopysparkCubeMetadata(dimension_names=['x', 'y', 't'])), 'dimension': 't', 'reducer': {'process_graph': {'mean1': {'process_id': 'mean', 'arguments': {'data': {'from_parameter': 'data'}}, 'result': True, '_node_id': 'mean1'}}}}  (ref: r-2609100535174fa1871cdf8f67cd0e95)

In [ ]:
# Smaller test area around Veerapandi
lat = 9.966636
lon = 77.429217

test_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": lon - 0.002,
        "south": lat - 0.002,
        "east": lon + 0.002,
        "north": lat + 0.002
    },
    temporal_extent=["2025-06-01", "2025-06-30"],
    bands=["B03", "B04", "B08"]
)

print("Sentinel-2 test data cube created!")

Sentinel-2 test data cube created!


In [ ]:
ndvi_test = (test_cube.band("B08") - test_cube.band("B04")) / (
    test_cube.band("B08") + test_cube.band("B04")
)

ndwi_test = (test_cube.band("B03") - test_cube.band("B08")) / (
    test_cube.band("B03") + test_cube.band("B08")
)

print("NDVI and NDWI created successfully!")

NDVI and NDWI created successfully!


In [ ]:
from openeo.processes import mean

# Small area around the check dam
roi = {
    "type": "Polygon",
    "coordinates": [[
        [lon-0.0005, lat-0.0005],
        [lon+0.0005, lat-0.0005],
        [lon+0.0005, lat+0.0005],
        [lon-0.0005, lat+0.0005],
        [lon-0.0005, lat-0.0005]
    ]]
}

ndvi_value = ndvi_test.aggregate_spatial(
    geometries=roi,
    reducer=mean
)

ndwi_value = ndwi_test.aggregate_spatial(
    geometries=roi,
    reducer=mean
)

print("ROI created. Ready to get actual Sentinel-2 values.")

ROI created. Ready to get actual Sentinel-2 values.


In [ ]:
ndvi_result = ndvi_value.execute()

print("NDVI Result:")
print(ndvi_result)

NDVI Result:
{'2025-06-01T00:00:00Z': [[0.5112220277347841]], '2025-06-03T00:00:00Z': [[0.4867929570946442]], '2025-06-06T00:00:00Z': [[0.4980684931366896]], '2025-06-11T00:00:00Z': [[0.1974660498051604]], '2025-06-16T00:00:00Z': [[0.0307600529294861]], '2025-06-21T00:00:00Z': [[0.4684663794334393]], '2025-06-23T00:00:00Z': [[0.0224109501847304]], '2025-06-26T00:00:00Z': [[0.0025103917049074]]}


In [ ]:
ndwi_result = ndwi_value.execute()

print("NDWI Result:")
print(ndwi_result)

NDWI Result:
{'2025-06-01T00:00:00Z': [[-0.5048825109054235]], '2025-06-03T00:00:00Z': [[-0.4651239772735179]], '2025-06-06T00:00:00Z': [[-0.4991304179853644]], '2025-06-11T00:00:00Z': [[-0.1773456357234765]], '2025-06-16T00:00:00Z': [[-0.0355542546738524]], '2025-06-21T00:00:00Z': [[-0.4371376520192081]], '2025-06-23T00:00:00Z': [[-0.014197994500768]], '2025-06-26T00:00:00Z': [[0.0075276567714196]]}


In [ ]:
before_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": lon - 0.002,
        "south": lat - 0.002,
        "east": lon + 0.002,
        "north": lat + 0.002
    },
    temporal_extent=["2020-06-01", "2020-06-30"],
    bands=["B03", "B04", "B08"]
)

print("2020 Before Sentinel-2 data cube created!")

2020 Before Sentinel-2 data cube created!


In [ ]:
before_ndvi = (
    before_cube.band("B08") - before_cube.band("B04")
) / (
    before_cube.band("B08") + before_cube.band("B04")
)

before_ndwi = (
    before_cube.band("B03") - before_cube.band("B08")
) / (
    before_cube.band("B03") + before_cube.band("B08")
)

print("2020 Before NDVI and NDWI created successfully!")

2020 Before NDVI and NDWI created successfully!


In [ ]:
before_ndvi_value = before_ndvi.aggregate_spatial(
    geometries=roi,
    reducer=mean
)

before_ndvi_result = before_ndvi_value.execute()

print("2020 Before NDVI Result:")
print(before_ndvi_result)

2020 Before NDVI Result:
{'2020-06-02T00:00:00Z': [[0.405154289169745]], '2020-06-07T00:00:00Z': [[0.031372260764118]], '2020-06-12T00:00:00Z': [[0.0564232700742965]], '2020-06-17T00:00:00Z': [[0.1886029955276773]], '2020-06-22T00:00:00Z': [[0.0907622201999357]], '2020-06-27T00:00:00Z': [[0.1134432691557348]]}


In [ ]:
before_ndwi_value = before_ndwi.aggregate_spatial(
    geometries=roi,
    reducer=mean
)

before_ndwi_result = before_ndwi_value.execute()

print("2020 Before NDWI Result:")
print(before_ndwi_result)

2020 Before NDWI Result:
{'2020-06-02T00:00:00Z': [[-0.4017172498890191]], '2020-06-07T00:00:00Z': [[0.0137892696998767]], '2020-06-12T00:00:00Z': [[-0.0538489067357433]], '2020-06-17T00:00:00Z': [[-0.1880784607376934]], '2020-06-22T00:00:00Z': [[-0.0958509249997533]], '2020-06-27T00:00:00Z': [[-0.113758578280772]]}


In [ ]:
import numpy as np

# Extract values
before_ndvi_vals = [
    0.405154289169745,
    0.031372260764118,
    0.0564232700742965,
    0.1886029955276773,
    0.0907622201999357,
    0.1134432691557348
]

after_ndvi_vals = [
    0.5112220277347841,
    0.4867929570946442,
    0.4980684931366896,
    0.1974660498051604,
    0.0307600529294861,
    0.4684663794334393,
    0.0224109501847304,
    0.0025103917049074
]

before_ndwi_vals = [
    -0.4017172498890191,
    0.0137892696998767,
    -0.0538489067357433,
    -0.1880784607376934,
    -0.0958509249997533,
    -0.113758578280772
]

after_ndwi_vals = [
    -0.5048825109054235,
    -0.4651239772735179,
    -0.4991304179853644,
    -0.1773456357234765,
    -0.0355542546738524,
    -0.4371376520192081,
    -0.014197994500768,
    0.0075276567714196
]

before_ndvi = np.mean(before_ndvi_vals)
after_ndvi = np.mean(after_ndvi_vals)

before_ndwi = np.mean(before_ndwi_vals)
after_ndwi = np.mean(after_ndwi_vals)

print("VEERAPANDI")
print("-------------------------")
print(f"Before NDVI : {before_ndvi:.4f}")
print(f"After NDVI  : {after_ndvi:.4f}")
print(f"NDVI Change : {after_ndvi-before_ndvi:+.4f}")
print("NDVI Status :", "Increased" if after_ndvi > before_ndvi else "Decreased")

print()

print(f"Before NDWI : {before_ndwi:.4f}")
print(f"After NDWI  : {after_ndwi:.4f}")
print(f"NDWI Change : {after_ndwi-before_ndwi:+.4f}")
print("NDWI Status :", "Increased" if after_ndwi > before_ndwi else "Decreased")

VEERAPANDI
-------------------------
Before NDVI : 0.1476
After NDVI  : 0.2772
NDVI Change : +0.1296
NDVI Status : Increased

Before NDWI : -0.1399
After NDWI  : -0.2657
NDWI Change : -0.1258
NDWI Status : Decreased


In [ ]:
lat2 = 13.261128
lon2 = 80.168365

after2_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": lon2 - 0.002,
        "south": lat2 - 0.002,
        "east": lon2 + 0.002,
        "north": lat2 + 0.002
    },
    temporal_extent=["2025-06-01", "2025-06-30"],
    bands=["B03", "B04", "B08"]
)

print("Jagannathapuram 2025 After cube created!")

Jagannathapuram 2025 After cube created!


In [ ]:
after2_ndvi = (
    after2_cube.band("B08") - after2_cube.band("B04")
) / (
    after2_cube.band("B08") + after2_cube.band("B04")
)

after2_ndwi = (
    after2_cube.band("B03") - after2_cube.band("B08")
) / (
    after2_cube.band("B03") + after2_cube.band("B08")
)

print("Jagannathapuram 2025 NDVI and NDWI created!")

Jagannathapuram 2025 NDVI and NDWI created!


In [ ]:
roi2 = {
    "type": "Polygon",
    "coordinates": [[
        [lon2-0.0005, lat2-0.0005],
        [lon2+0.0005, lat2-0.0005],
        [lon2+0.0005, lat2+0.0005],
        [lon2-0.0005, lat2+0.0005],
        [lon2-0.0005, lat2-0.0005]
    ]]
}

after2_ndvi_value = after2_ndvi.aggregate_spatial(
    geometries=roi2,
    reducer=mean
)

after2_ndvi_result = after2_ndvi_value.execute()

print("Jagannathapuram 2025 NDVI:")
print(after2_ndvi_result)

Jagannathapuram 2025 NDVI:
{'2025-06-03T00:00:00Z': [[0.4082694089055554]], '2025-06-08T00:00:00Z': [[0.4103655740618706]], '2025-06-13T00:00:00Z': [[0.4576652060640006]], '2025-06-18T00:00:00Z': [[0.3629543296069153]], '2025-06-20T00:00:00Z': [[0.4430397441071912]], '2025-06-23T00:00:00Z': [[0.069090296660573]], '2025-06-28T00:00:00Z': [[0.1298266316740966]]}


In [ ]:
after2_ndwi_value = after2_ndwi.aggregate_spatial(
    geometries=roi2,
    reducer=mean
)

after2_ndwi_result = after2_ndwi_value.execute()

print("Jagannathapuram 2025 NDWI:")
print(after2_ndwi_result)

Jagannathapuram 2025 NDWI:
{'2025-06-03T00:00:00Z': [[-0.3348816848794876]], '2025-06-08T00:00:00Z': [[-0.3509904134957012]], '2025-06-13T00:00:00Z': [[-0.3696027140571805]], '2025-06-18T00:00:00Z': [[-0.3234554987308408]], '2025-06-20T00:00:00Z': [[-0.4098944647376202]], '2025-06-23T00:00:00Z': [[-0.0655033233121406]], '2025-06-28T00:00:00Z': [[-0.0538671735889655]]}


In [ ]:
before2_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": lon2 - 0.002,
        "south": lat2 - 0.002,
        "east": lon2 + 0.002,
        "north": lat2 + 0.002
    },
    temporal_extent=["2020-06-01", "2020-06-30"],
    bands=["B03", "B04", "B08"]
)

print("Jagannathapuram 2020 Before cube created!")

Jagannathapuram 2020 Before cube created!


In [ ]:
before2_ndvi = (
    before2_cube.band("B08") - before2_cube.band("B04")
) / (
    before2_cube.band("B08") + before2_cube.band("B04")
)

before2_ndwi = (
    before2_cube.band("B03") - before2_cube.band("B08")
) / (
    before2_cube.band("B03") + before2_cube.band("B08")
)

print("Jagannathapuram 2020 NDVI and NDWI created!")

Jagannathapuram 2020 NDVI and NDWI created!


In [ ]:
before2_ndvi_value = before2_ndvi.aggregate_spatial(
    geometries=roi2,
    reducer=mean
)

before2_ndvi_result = before2_ndvi_value.execute()

print("Jagannathapuram 2020 NDVI:")
print(before2_ndvi_result)

Jagannathapuram 2020 NDVI:
{'2020-06-04T00:00:00Z': [[0.5233556743988321]], '2020-06-09T00:00:00Z': [[0.0620011236857284]], '2020-06-14T00:00:00Z': [[0.3160792807163285]], '2020-06-19T00:00:00Z': [[0.3030801018900123]], '2020-06-24T00:00:00Z': [[0.320494603088572]], '2020-06-29T00:00:00Z': [[0.0436894740940125]]}


In [ ]:
before2_ndwi_result = before2_ndwi.aggregate_spatial(
    geometries=roi2,
    reducer=mean
).execute()

print("Jagannathapuram 2020 NDWI:")
print(before2_ndwi_result)

Jagannathapuram 2020 NDWI:
{'2020-06-04T00:00:00Z': [[-0.499963558656125]], '2020-06-09T00:00:00Z': [[-0.0568920006313599]], '2020-06-14T00:00:00Z': [[-0.3965198547879526]], '2020-06-19T00:00:00Z': [[-0.4053538732292238]], '2020-06-24T00:00:00Z': [[-0.4042064732263896]], '2020-06-29T00:00:00Z': [[-0.0373659907601589]]}


In [ ]:
import numpy as np

# Jagannathapuram NDVI
before_ndvi2 = np.mean([
    0.5233556743988321,
    0.0620011236857284,
    0.3160792807163285,
    0.3030801018900123,
    0.320494603088572,
    0.0436894740940125
])

after_ndvi2 = np.mean([
    0.4082694089055554,
    0.4103655740618706,
    0.4576652060640006,
    0.3629543296069153,
    0.4430397441071912,
    0.069090296660573,
    0.1298266316740966
])

# Jagannathapuram NDWI
before_ndwi2 = np.mean([
    -0.499963558656125,
    -0.0568920006313599,
    -0.3965198547879526,
    -0.4053538732292238,
    -0.4042064732263896,
    -0.0373659907601589
])

after_ndwi2 = np.mean([
    -0.3348816848794876,
    -0.3509904134957012,
    -0.3696027140571805,
    -0.3234554987308408,
    -0.4098944647376202,
    -0.0655033233121406,
    -0.0538671735889655
])

print("Jagannathapuram")
print("Before NDVI:", before_ndvi2)
print("After NDVI :", after_ndvi2)
print("NDVI Change:", after_ndvi2 - before_ndvi2)

print("Before NDWI:", before_ndwi2)
print("After NDWI :", after_ndwi2)
print("NDWI Change:", after_ndwi2 - before_ndwi2)

Jagannathapuram
Before NDVI: 0.26145004297891433
After NDVI : 0.3258873130114575
NDVI Change: 0.06443727003254318
Before NDWI: -0.30005029188186827
After NDWI : -0.27259932468599085
NDWI Change: 0.027450967195877418


In [ ]:
lat3 = 11.844412
lon3 = 79.735178

after3_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": lon3 - 0.002,
        "south": lat3 - 0.002,
        "east": lon3 + 0.002,
        "north": lat3 + 0.002
    },
    temporal_extent=["2025-06-01", "2025-06-30"],
    bands=["B03", "B04", "B08"]
)

after3_ndvi = (
    after3_cube.band("B08") - after3_cube.band("B04")
) / (
    after3_cube.band("B08") + after3_cube.band("B04")
)

after3_ndwi = (
    after3_cube.band("B03") - after3_cube.band("B08")
) / (
    after3_cube.band("B03") + after3_cube.band("B08")
)

print("Cuddalore 2025 data loaded successfully")

Cuddalore 2025 data loaded successfully


In [ ]:
from openeo.processes import mean

roi3 = {
    "type": "Polygon",
    "coordinates": [[
        [lon3-0.0005, lat3-0.0005],
        [lon3+0.0005, lat3-0.0005],
        [lon3+0.0005, lat3+0.0005],
        [lon3-0.0005, lat3+0.0005],
        [lon3-0.0005, lat3-0.0005]
    ]]
}

after3_ndvi_result = after3_ndvi.aggregate_spatial(
    geometries=roi3,
    reducer=mean
).execute()

after3_ndwi_result = after3_ndwi.aggregate_spatial(
    geometries=roi3,
    reducer=mean
).execute()

print("Cuddalore 2025 NDVI:")
print(after3_ndvi_result)

print("\nCuddalore 2025 NDWI:")
print(after3_ndwi_result)

Cuddalore 2025 NDVI:
{'2025-06-03T00:00:00Z': [[0.5837943880272306]], '2025-06-08T00:00:00Z': [[0.602782718467811]], '2025-06-13T00:00:00Z': [[0.6419909713435764]], '2025-06-18T00:00:00Z': [[0.466988352335189]], '2025-06-20T00:00:00Z': [[0.6110210769678935]], '2025-06-23T00:00:00Z': [[0.1252724771534115]], '2025-06-28T00:00:00Z': [[0.6266943740327496]]}

Cuddalore 2025 NDWI:
{'2025-06-03T00:00:00Z': [[-0.5282549711544652]], '2025-06-08T00:00:00Z': [[-0.5486885250107316]], '2025-06-13T00:00:00Z': [[-0.5620046145906133]], '2025-06-18T00:00:00Z': [[-0.4250938298781056]], '2025-06-20T00:00:00Z': [[-0.5601200184299926]], '2025-06-23T00:00:00Z': [[-0.1110988322122037]], '2025-06-28T00:00:00Z': [[-0.5750367459186838]]}


In [ ]:
before3_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": lon3 - 0.002,
        "south": lat3 - 0.002,
        "east": lon3 + 0.002,
        "north": lat3 + 0.002
    },
    temporal_extent=["2020-06-01", "2020-06-30"],
    bands=["B03", "B04", "B08"]
)

before3_ndvi = (
    before3_cube.band("B08") - before3_cube.band("B04")
) / (
    before3_cube.band("B08") + before3_cube.band("B04")
)

before3_ndwi = (
    before3_cube.band("B03") - before3_cube.band("B08")
) / (
    before3_cube.band("B03") + before3_cube.band("B08")
)

print("Cuddalore 2020 data loaded successfully")

Cuddalore 2020 data loaded successfully


In [ ]:
before3_ndvi_result = before3_ndvi.aggregate_spatial(
    geometries=roi3,
    reducer=mean
).execute()

before3_ndwi_result = before3_ndwi.aggregate_spatial(
    geometries=roi3,
    reducer=mean
).execute()

print("Cuddalore 2020 NDVI:")
print(before3_ndvi_result)

print("\nCuddalore 2020 NDWI:")
print(before3_ndwi_result)

Cuddalore 2020 NDVI:
{'2020-06-04T00:00:00Z': [[0.5569477586214208]], '2020-06-09T00:00:00Z': [[0.3361023766689064]], '2020-06-14T00:00:00Z': [[0.5615167401053689]], '2020-06-19T00:00:00Z': [[0.0661317881596975]], '2020-06-24T00:00:00Z': [[0.5102804516834661]], '2020-06-29T00:00:00Z': [[0.1974089799587391]]}

Cuddalore 2020 NDWI:
{'2020-06-04T00:00:00Z': [[-0.5478995368500387]], '2020-06-09T00:00:00Z': [[-0.3334684007423968]], '2020-06-14T00:00:00Z': [[-0.5554715865407108]], '2020-06-19T00:00:00Z': [[-0.0169772339045755]], '2020-06-24T00:00:00Z': [[-0.4974140055662344]], '2020-06-29T00:00:00Z': [[-0.1811505344288408]]}


In [ ]:
import numpy as np

# Cuddalore NDVI
before_ndvi3 = np.mean([
    0.5569477586214208,
    0.3361023766689064,
    0.5615167401053689,
    0.0661317881596975,
    0.5102804516834661,
    0.1974089799587391
])

after_ndvi3 = np.mean([
    0.5837943880272306,
    0.602782718467811,
    0.6419909713435764,
    0.466988352335189,
    0.6110210769678935,
    0.1252724771534115,
    0.6266943740327496
])

# Cuddalore NDWI
before_ndwi3 = np.mean([
    -0.5478995368500387,
    -0.3334684007423968,
    -0.5554715865407108,
    -0.0169772339045755,
    -0.4974140055662344,
    -0.1811505344288408
])

after_ndwi3 = np.mean([
    -0.5282549711544652,
    -0.5486885250107316,
    -0.5620046145906133,
    -0.4250938298781056,
    -0.5601200184299926,
    -0.1110988322122037,
    -0.5750367459186838
])

print("Cuddalore")
print("Before NDVI:", before_ndvi3)
print("After NDVI :", after_ndvi3)
print("NDVI Change:", after_ndvi3 - before_ndvi3)

print("Before NDWI:", before_ndwi3)
print("After NDWI :", after_ndwi3)
print("NDWI Change:", after_ndwi3 - before_ndwi3)

Cuddalore
Before NDVI: 0.3713980158662665
After NDVI : 0.5226491940468374
NDVI Change: 0.15125117818057093
Before NDWI: -0.3553968830054661
After NDWI : -0.4728996481706852
NDWI Change: -0.11750276516521907


In [ ]:
lat4 = 11.805000
lon4 = 79.659944

after4_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": lon4 - 0.002,
        "south": lat4 - 0.002,
        "east": lon4 + 0.002,
        "north": lat4 + 0.002
    },
    temporal_extent=["2025-06-01", "2025-06-30"],
    bands=["B03", "B04", "B08"]
)

after4_ndvi = (
    after4_cube.band("B08") - after4_cube.band("B04")
) / (
    after4_cube.band("B08") + after4_cube.band("B04")
)

after4_ndwi = (
    after4_cube.band("B03") - after4_cube.band("B08")
) / (
    after4_cube.band("B03") + after4_cube.band("B08")
)

print("Viswanthapuram 2025 data loaded successfully")

Viswanthapuram 2025 data loaded successfully


In [ ]:
from openeo.processes import mean

roi4 = {
    "type": "Polygon",
    "coordinates": [[
        [lon4-0.0005, lat4-0.0005],
        [lon4+0.0005, lat4-0.0005],
        [lon4+0.0005, lat4+0.0005],
        [lon4-0.0005, lat4+0.0005],
        [lon4-0.0005, lat4-0.0005]
    ]]
}

after4_ndvi_result = after4_ndvi.aggregate_spatial(
    geometries=roi4,
    reducer=mean
).execute()

after4_ndwi_result = after4_ndwi.aggregate_spatial(
    geometries=roi4,
    reducer=mean
).execute()

print("Viswanthapuram 2025 NDVI:")
print(after4_ndvi_result)

print("\nViswanthapuram 2025 NDWI:")
print(after4_ndwi_result)

Viswanthapuram 2025 NDVI:
{'2025-06-03T00:00:00Z': [[0.4644236542955041]], '2025-06-08T00:00:00Z': [[0.4891650205850601]], '2025-06-13T00:00:00Z': [[0.5350224033296108]], '2025-06-18T00:00:00Z': [[0.4072174081802368]], '2025-06-20T00:00:00Z': [[0.5229431335628033]], '2025-06-23T00:00:00Z': [[0.0989651920199394]], '2025-06-28T00:00:00Z': [[0.5318352288007736]]}

Viswanthapuram 2025 NDWI:
{'2025-06-03T00:00:00Z': [[-0.4457554753273725]], '2025-06-08T00:00:00Z': [[-0.4951331284344196]], '2025-06-13T00:00:00Z': [[-0.500013257086277]], '2025-06-18T00:00:00Z': [[-0.4044360336065292]], '2025-06-20T00:00:00Z': [[-0.5016792076826095]], '2025-06-23T00:00:00Z': [[-0.0994949840903282]], '2025-06-28T00:00:00Z': [[-0.5144953157901764]]}


In [ ]:
before4_ndvi_result = before4_ndvi.aggregate_spatial(
    geometries=roi4,
    reducer=mean
).execute()

before4_ndwi_result = before4_ndwi.aggregate_spatial(
    geometries=roi4,
    reducer=mean
).execute()

print("Viswanthapuram 2020 NDVI:")
print(before4_ndvi_result)

print("\nViswanthapuram 2020 NDWI:")
print(before4_ndwi_result)

Viswanthapuram 2020 NDVI:
{'2020-06-04T00:00:00Z': [[0.426696217238903]], '2020-06-09T00:00:00Z': [[0.3303530280590057]], '2020-06-14T00:00:00Z': [[0.4797599318921566]], '2020-06-19T00:00:00Z': [[0.1383295346498489]], '2020-06-24T00:00:00Z': [[0.4283416420221329]], '2020-06-29T00:00:00Z': [[0.1751095073223114]]}

Viswanthapuram 2020 NDWI:
{'2020-06-04T00:00:00Z': [[-0.451008141040802]], '2020-06-09T00:00:00Z': [[-0.3378780944347381]], '2020-06-14T00:00:00Z': [[-0.4740628781318664]], '2020-06-19T00:00:00Z': [[-0.104779689669609]], '2020-06-24T00:00:00Z': [[-0.410122022986412]], '2020-06-29T00:00:00Z': [[-0.1509551906585693]]}


In [ ]:
import numpy as np

before_ndvi4 = np.mean([
    0.426696217238903,
    0.3303530280590057,
    0.4797599318921566,
    0.1383295346498489,
    0.4283416420221329,
    0.1751095073223114
])

after_ndvi4 = np.mean([
    0.4644236542955041,
    0.4891650205850601,
    0.5350224033296108,
    0.4072174081802368,
    0.5229431335628033,
    0.0989651920199394,
    0.5318352288007736
])

before_ndwi4 = np.mean([
    -0.451008141040802,
    -0.3378780944347381,
    -0.4740628781318664,
    -0.104779689669609,
    -0.410122022986412,
    -0.1509551906585693
])

after_ndwi4 = np.mean([
    -0.4457554753273725,
    -0.4951331284344196,
    -0.500013257086277,
    -0.4044360336065292,
    -0.5016792076826095,
    -0.0994949840903282,
    -0.5144953157901764
])

print("Viswanthapuram")
print("Before NDVI:", before_ndvi4)
print("After NDVI :", after_ndvi4)
print("NDVI Change:", after_ndvi4 - before_ndvi4)

print("Before NDWI:", before_ndwi4)
print("After NDWI :", after_ndwi4)
print("NDWI Change:", after_ndwi4 - before_ndwi4)

Viswanthapuram
Before NDVI: 0.32976497686405976
After NDVI : 0.4356531486819897
NDVI Change: 0.10588817181792992
Before NDWI: -0.32146766948699945
After NDWI : -0.4230010574311018
NDWI Change: -0.10153338794410233


In [ ]:
lat5 = 16.495961
lon5 = 77.894172

after5_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": lon5 - 0.002,
        "south": lat5 - 0.002,
        "east": lon5 + 0.002,
        "north": lat5 + 0.002
    },
    temporal_extent=["2025-06-01", "2025-06-30"],
    bands=["B03", "B04", "B08"]
)

after5_ndvi = (
    after5_cube.band("B08") - after5_cube.band("B04")
) / (
    after5_cube.band("B08") + after5_cube.band("B04")
)

after5_ndwi = (
    after5_cube.band("B03") - after5_cube.band("B08")
) / (
    after5_cube.band("B03") + after5_cube.band("B08")
)

print("Peddamungalachedu 2025 data loaded successfully")

Peddamungalachedu 2025 data loaded successfully


In [ ]:
roi5 = {
    "type": "Polygon",
    "coordinates": [[
        [lon5-0.0005, lat5-0.0005],
        [lon5+0.0005, lat5-0.0005],
        [lon5+0.0005, lat5+0.0005],
        [lon5-0.0005, lat5+0.0005],
        [lon5-0.0005, lat5-0.0005]
    ]]
}

after5_ndvi_result = after5_ndvi.aggregate_spatial(
    geometries=roi5,
    reducer=mean
).execute()

after5_ndwi_result = after5_ndwi.aggregate_spatial(
    geometries=roi5,
    reducer=mean
).execute()

print("Peddamungalachedu 2025 NDVI:")
print(after5_ndvi_result)

print("\nPeddamungalachedu 2025 NDWI:")
print(after5_ndwi_result)

Peddamungalachedu 2025 NDVI:
{'2025-06-01T00:00:00Z': [[-0.0259770846166571]], '2025-06-03T00:00:00Z': [[0.193780913671925]], '2025-06-06T00:00:00Z': [[0.3023401311606415]], '2025-06-11T00:00:00Z': [[-0.0325098726128743]], '2025-06-16T00:00:00Z': [[0.0192504368957193]], '2025-06-21T00:00:00Z': [[0.3353690189147784]], '2025-06-23T00:00:00Z': [[0.0593325375032819]], '2025-06-26T00:00:00Z': [[0.0454143355023269]]}

Peddamungalachedu 2025 NDWI:
{'2025-06-01T00:00:00Z': [[0.0651832302447316]], '2025-06-03T00:00:00Z': [[-0.1946830713798192]], '2025-06-06T00:00:00Z': [[-0.3283964160306394]], '2025-06-11T00:00:00Z': [[0.057285709475929]], '2025-06-16T00:00:00Z': [[0.016908456478316]], '2025-06-21T00:00:00Z': [[-0.3553006014794357]], '2025-06-23T00:00:00Z': [[-0.058689803500806]], '2025-06-26T00:00:00Z': [[-0.0436770637050147]]}


In [ ]:
before5_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": lon5 - 0.002,
        "south": lat5 - 0.002,
        "east": lon5 + 0.002,
        "north": lat5 + 0.002
    },
    temporal_extent=["2020-06-01", "2020-06-30"],
    bands=["B03", "B04", "B08"]
)

before5_ndvi = (
    before5_cube.band("B08") - before5_cube.band("B04")
) / (
    before5_cube.band("B08") + before5_cube.band("B04")
)

before5_ndwi = (
    before5_cube.band("B03") - before5_cube.band("B08")
) / (
    before5_cube.band("B03") + before5_cube.band("B08")
)

print("Peddamungalachedu 2020 data loaded successfully")

Peddamungalachedu 2020 data loaded successfully


In [ ]:
before5_ndvi_result = before5_ndvi.aggregate_spatial(
    geometries=roi5,
    reducer=mean
).execute()

before5_ndwi_result = before5_ndwi.aggregate_spatial(
    geometries=roi5,
    reducer=mean
).execute()

print("Peddamungalachedu 2020 NDVI:")
print(before5_ndvi_result)

print("\nPeddamungalachedu 2020 NDWI:")
print(before5_ndwi_result)

Peddamungalachedu 2020 NDVI:
{'2020-06-02T00:00:00Z': [[0.2218988860903446]], '2020-06-07T00:00:00Z': [[0.198654518947621]], '2020-06-12T00:00:00Z': [[0.0213890576288719]], '2020-06-17T00:00:00Z': [[-0.0271951246021454]], '2020-06-22T00:00:00Z': [[0.0587962442654962]], '2020-06-27T00:00:00Z': [[0.2244348721809623]]}

Peddamungalachedu 2020 NDWI:
{'2020-06-02T00:00:00Z': [[-0.302064364729834]], '2020-06-07T00:00:00Z': [[-0.2872203455483618]], '2020-06-12T00:00:00Z': [[-0.0265204727742051]], '2020-06-17T00:00:00Z': [[0.0262816201064212]], '2020-06-22T00:00:00Z': [[-0.0172404737598051]], '2020-06-27T00:00:00Z': [[-0.280737448452918]]}


In [ ]:
import numpy as np

before_ndvi5 = np.mean([
    0.2218988860903446,
    0.198654518947621,
    0.0213890576288719,
    -0.0271951246021454,
    0.0587962442654962,
    0.2244348721809623
])

after_ndvi5 = np.mean([
    -0.0259770846166571,
    0.193780913671925,
    0.3023401311606415,
    -0.0325098726128743,
    0.0192504368957193,
    0.3353690189147784,
    0.0593325375032819,
    0.0454143355023269
])

before_ndwi5 = np.mean([
    -0.302064364729834,
    -0.2872203455483618,
    -0.0265204727742051,
    0.0262816201064212,
    -0.0172404737598051,
    -0.280737448452918
])

after_ndwi5 = np.mean([
    0.0651832302447316,
    -0.1946830713798192,
    -0.3283964160306394,
    0.057285709475929,
    0.016908456478316,
    -0.3553006014794357,
    -0.058689803500806,
    -0.0436770637050147
])

print("Peddamungalachedu")
print("Before NDVI:", before_ndvi5)
print("After NDVI :", after_ndvi5)
print("NDVI Change:", after_ndvi5 - before_ndvi5)

print("Before NDWI:", before_ndwi5)
print("After NDWI :", after_ndwi5)
print("NDWI Change:", after_ndwi5 - before_ndwi5)

Peddamungalachedu
Before NDVI: 0.1163297424185251
After NDVI : 0.1121250520523927
NDVI Change: -0.004204690366132399
Before NDWI: -0.14791691419311712
After NDWI : -0.10517119498709229
NDWI Change: 0.042745719206024824
